# Session 13：需要时才打开的练习素材库

这不是今天一定要跑完的课程。先和导师谈谈最近的作业，以及你希望得到什么帮助，再挑一个用得上的例子。

本档使用合成数据，不含真实学生或商店数据。每一行代表一家门店的一年；sales 和 profit 的单位都设定为万元人民币。

## 如何挑选

- 只想看懂表格：模块 0 → 1。
- 想检查数据：模块 0 → 1 → 2，三种检查可分开看。
- 想学筛选：模块 0 → 1 → 4。
- 想学分组加总：模块 0 → 1 → 5，不必先做筛选。
- 想画五年合计图：模块 0 → 1 → 5 → 6A → 6B。
- 想做 2025 比较或趋势题：模块 0 → 1 → 6A，再选 7 或 8。
- 模块 3 是选读。只有想分清 Series 和 DataFrame 时才看。

可以跳过主题，但仍需先执行列出的前置格。后面的格子会使用前面建立的变量，Notebook 不会自动替你补跑。

每段都分成「先做什么」「应看到什么」「为什么」「如何确认」。不必每一步都猜结果；先弄清楚现在想回答的问题就好。

## 模块 0：确认现在使用的 Python 与文件位置

**先做什么：** 选好 Python kernel，执行下面这一格。请先下载完整 session13 文件夹，保留其中的 data 子目录。

**应看到什么：** Python 的执行路径、pandas 与 matplotlib 的版本，以及 CSV 和输出文件夹的位置。

**为什么：** Notebook 使用的 Python 不一定和终端机相同。我们先确认这个 Python 能找到需要的套件和文件。

**如何确认：** 没有红色报错，且 CSV 路径指向本课的 sample-sales.csv。这只代表环境格成功，不代表后面的分析已经完成。

如果看到「No module named pandas」，表示目前这个 Python 找不到 pandas。请先和导师确认环境，再处理安装；不用反复重新建立 notebook。

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("Python 路径：", sys.executable)
print("pandas 版本：", pd.__version__)
print("matplotlib 版本：", matplotlib.__version__)
print("目前工作目录：", Path.cwd())

# 从 session13 文件夹或仓库根目录开启，都可以找到同一份 CSV。
lesson_dir = Path.cwd()
if not (lesson_dir / "data/sample-sales.csv").is_file():
    lesson_dir = lesson_dir / "session13"
csv_path = lesson_dir / "data/sample-sales.csv"
if not csv_path.is_file():
    raise FileNotFoundError("找不到 CSV。请从 session13 或仓库根目录开启，并确认 data 文件夹已下载。")
output_dir = lesson_dir / "outputs"
output_dir.mkdir(exist_ok=True)
print("CSV 位置：", csv_path.resolve())
print("图片输出位置：", output_dir.resolve())

## 模块 1：把 CSV 读成一张表

前置：模块 0。

**先做什么：** 执行下面一格。

**应看到什么：** 第一段输出是 `(20, 5)`，表示整张表有 20 行、5 栏。接下来只显示前 5 行，方便先看数据长相。

**为什么：** `df.shape` 查看整张表的大小；`df.head()` 预览前 5 行，没有删除其余 15 行。

**如何确认：** 预览中的前 5 行都是门店 A，年份从 2021 到 2025。完整数据仍有 A、B、C、D 四家店。

`pd` 是我们替 pandas 取的简称。`df` 是用来存这张表的变量名，并不是 Python 规定的名字。也可以取名 sales_table，但后面使用它的地方就要一致改名。先不需要为此改代码。

In [ ]:
df = pd.read_csv(csv_path)
print("整张表的行数、栏数：", df.shape)
print("下面只预览前 5 行：")
print(df.head().to_string(index=False))

### 先读懂一行，再继续分析

第一行表示：门店 A 在 2021 年的销售额是 120 万元，利润是 18 万元。

category 栏的「电子」只是该店当年的主力品类标记。它没有告诉我们电子商品单独赚了多少，因此不能把这行的 18 万元直接解读为「电子类利润」。

如果现在需要的只是理解数据，到这里就可以回到自己的问题。

## 模块 2：检查数据，三份报告分开看

前置：模块 0、1。只有需要检查数据时才做，不用一次读完三份报告。

### 2A：每个字段目前被当成哪一种数据？

**先做什么：** 执行下一格。

**应看到什么：** year、sales、profit 通常显示 int64，意思是整数；store 和 category 在不同 pandas 版本可能显示 object 或 str，这份样本的内容是文字。

**为什么：** dtype 是「数据类型」的简称。数字若被读成文字，后面计算可能不符合预期。object 本身也不保证每个值都是文字，仍要看内容。

**如何确认：** 年份、销售额、利润是可以计算的数字。这份报告不会同时告诉你是否有缺值或重复，下面是另外两种检查。

In [ ]:
print(df.dtypes)

### 2B：有多少格没有数据？

**先做什么：** 执行下一格。

**应看到什么：** 每个字段后面都是 0，表示本样本没有 pandas 判定为缺值的格子。

**为什么：** `isna()` 找缺值，再用 `sum()` 计算每栏有几个。这里加的是缺值标记，不是利润。

**如何确认：** 看完整报告，不只看 head 的前 5 行。真实数据若出现缺值，先问它代表什么，不直接填成 0。空白字串或其他特殊标记也可能需要另外确认。

In [ ]:
print(df.isna().sum())

### 2C：同一家店、同一年是否出现不只一行？

**先做什么：** 执行下一格。

**应看到什么：** 0，表示本样本没有重复的店年组合。

**为什么：** 这份数据约定一店一年只有一行，所以把 store 与 year 合在一起检查。若真实数据的一行是订单，同店同年有多行反而很正常。

**如何确认：** 先知道一行代表什么，再判断重复是不是问题。不要只因为看到重复就删除。

In [ ]:
print(df.duplicated(["store", "year"]).sum())

## 模块 3：只取一栏（选读）

前置：模块 0、1。这段只在你想知道两种括号差别时使用，不影响你先学筛选或分组。

**先做什么：** 执行下一格。

**应看到什么：** 第一个结果是 Series，shape 为 `(20,)`；第二个是 DataFrame，shape 为 `(20, 1)`。

**为什么：** Series 是一串有标签的数据，这里仍然有 20 个利润值，并不是只有一个数字。DataFrame 是二维表，这里只有一栏、20 行。两者没有谁比较高阶。

**如何确认：** 两个版本都保留 20 笔数据，只是数据结构不同。本 CSV 栏名是小写 profit；Python 区分大小写，写成 Profit 会找不到这个字段。

In [ ]:
profit_column = df["profit"]
profit_table = df[["profit"]]
print(type(profit_column).__name__, profit_column.shape)
print(type(profit_table).__name__, profit_table.shape)

## 模块 4：只留下利润大于 20 万元的行

前置：模块 0、1。不需要先执行模块 2 或 3。

**先做什么：** 执行下一格，先看行数，再看留下来的数据。

**应看到什么：** filtered 有 9 行、5 栏；原本的 df 仍有 20 行、5 栏。画面下方为了方便阅读，只列出 store、year、profit 这 3 栏。

**为什么：** 第一句代码筛掉不符合条件的行。最后一句另外选出 3 栏来显示。筛行和选栏是两个不同动作；filtered 里仍有全部 5 栏。

**如何确认：** 门店 A 有 4 行符合，门店 C 有 5 行符合，合计 9 行。条件是大于 20，不包含等于 20；这份样本刚好没有利润等于 20 的行。

In [ ]:
filtered = df[df["profit"] > 20]
print("筛选后的行数、栏数：", filtered.shape)
print("原表的行数、栏数：", df.shape)
print("下面另外只显示 3 栏：")
print(filtered[["store", "year", "profit"]].to_string(index=False))

### 需要练一个改动时：把 20 改成 30

先问自己：「利润刚好 30 的行，会不会留下？」

把上面条件改成 `profit > 30`，只重跑那一格。应留下 5 行：门店 A 的 2025 年，以及门店 C 的 2021、2023、2024、2025 年。刚好 30 的行不会留下。

练完若要回到本页的标准结果，再改回 20。无论门槛怎么改，原始 df 都不会被这段代码覆写。

## 模块 5：把一家店的五行数据，加成一个总数

前置：模块 0、1。这一段使用完整的 df，不使用前面的 filtered。

### 5A：先只看门店 A

**先做什么：** 执行下面一格，读出门店 A 的五年利润。

**应看到什么：** 2021 到 2025 年分别是 18、22、25、30、35 万元。

**为什么：** 先把会被加在一起的原始行摊开，才知道总数代表什么。

**如何确认：** 用计算机或心算做一次 18 + 22 + 25 + 30 + 35，应得到 130 万元。

In [ ]:
store_a_rows = df[df["store"] == "门店A"]
print(store_a_rows[["year", "profit"]].to_string(index=False))

### 5B：四家店都做同样的加总

**先做什么：** 执行下一格。

**应看到什么：** 原表有 20 行，结果是 4 个总数：A = 130、B = 71、C = 193、D = 62。单位仍是万元。

**为什么：** `groupby("store")` 把同一家店的行放在一起，`["profit"]` 指定要算利润，`sum()` 将每组利润相加。每家店的五年变成一个数字，所以结果不再保留每一年的明细。

**如何确认：** 先对照刚才手算的 A = 130，再抽查 B 的 12 + 14 + 11 + 15 + 19 = 71。

注意：这回答的是「各店五年利润总和」。如果题目要 2025 年或逐年趋势，这张结果表就不适合直接拿去画图。

In [ ]:
totals = df.groupby("store")["profit"].sum()
print("原表行数：", len(df))
print("店别总数的个数：", len(totals))
print(totals)

### 5C：再做一个整体核对

**先做什么：** 执行下一格。

**应看到什么：** 两个合计都是 456。

**为什么：** 我们只是按店重新分组，没有刻意删掉任何行，所以四家店总数再相加，应等于原表全部利润相加。

**如何确认：** 两个数字一致是一项证据，但不能取代对题意的确认。例如题目其实只要 2025 年，算出相等的五年总和仍然答错问题。

In [ ]:
print("四家店的总数再相加：", totals.sum())
print("原表全部利润相加：", df["profit"].sum())

## 模块 6A：画图前的共用设定

前置：模块 0。只有需要画图时才跑；不要求记住字体设定代码。

**先做什么：** 执行下面一格。

**应看到什么：** 找到常见中文字体就使用它；找不到就显示「使用英文标签」。这不是分析错误。

**为什么：** 有些电脑没有可用的中文字体。英文备援可以先让图保持可读，不需要在课中下载字体。

**如何确认：** 之后画出的 Store A–D 对应门店 A–D；Profit (10,000 CNY) 表示利润，单位是万元人民币。

In [ ]:
from matplotlib import font_manager

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
cjk_font = next((name for name in ["Noto Sans CJK TC", "Noto Sans CJK SC", "Microsoft YaHei", "PingFang TC", "SimHei"] if name in available_fonts), None)
use_chinese = cjk_font is not None
if use_chinese:
    plt.rcParams["font.family"] = cjk_font
plt.rcParams["axes.unicode_minus"] = False
store_labels = {f"门店{letter}": f"Store {letter}" for letter in "ABCD"}
y_label = "利润（万元人民币）" if use_chinese else "Profit (10,000 CNY)"
x_label = "门店" if use_chinese else "Store"
print("图表字体：", cjk_font or "使用英文标签")

## 模块 6B：把五年合计划成条形图

前置：模块 0、1、5B、6A。先核对 totals，确定你要回答的问题就是「各店五年总和」。

**先做什么：** 执行下一格，然后打开输出路径中的 profit-total.png。

**应看到什么：** 四根柱子，分别对应 130、71、193、62 万元。标题写明是 2021–2025 的合计，以及这是合成数据。

**为什么：** 这张图只是呈现 totals 里的四个数字。画图不会替我们判断应该算总和、平均，还是只算某一年。

**如何确认：** 图上最高的是门店 C。标题、单位和四根柱子的数值应与结果表一致。图片先保存，再显示。

In [ ]:
plot_totals = totals if use_chinese else totals.rename(index=store_labels)
ax = plot_totals.plot(kind="bar", figsize=(7, 4), rot=0)
ax.set_title("各店 2021–2025 利润合计（合成数据）" if use_chinese else "Profit by store, 2021–2025 total (synthetic)")
ax.set_xlabel(x_label)
ax.set_ylabel(y_label)
fig = ax.get_figure()
fig.tight_layout()
bar_path = output_dir / "profit-total.png"
fig.savefig(bar_path, dpi=150, bbox_inches="tight")
print("图片已保存到：", bar_path.resolve())
plt.show()
plt.close(fig)

## 模块 7：只比较 2025 年（需要时才选）

前置：模块 0、1、6A。不需要先完成所有前面的主题。

**先做什么：** 先说「我要比较 2025 年各店利润」。在下面空白格完成筛选，先印出结果表，确认后才画图。需要 AI 帮忙时，一次只请它处理下一步。

**应看到什么：** 2025 年有四家店各一行：A = 35、B = 19、C = 48、D = 17 万元。图应呈现这四个值，不是五年总和。

**为什么：** 这次题目有年份条件，要先只留下 2025 年。

**如何确认：** 从原表抽查 C 店的 2025 年，应是 48。若要存图，使用 outputs/profit-2025.png，标题注明 2025 年、单位和合成数据。

以下空白格不会自动产出答案或图片。教师答案只是备援，不必一开始照抄。

In [ ]:
# 在这里完成目前需要的一小步，例如先筛出 2025 年并印出结果。


## 模块 8：看五年的变化（需要时才选）

前置：模块 0、1、6A。只有目前问题与时间趋势有关时，才做这题。

**先做什么：** 保留每家店每一年的利润，按年份从小到大排列。可以请 AI 协助把 year 放在行、store 放在栏、profit 放在格子中。

**应看到什么：** 一张 5 行、4 栏的结果表；画出来有四条线，每条线五个年份点。

**为什么：** 要看变化就必须保留每年数字，不能使用只剩四个五年总数的 totals。

**如何确认：** B 店依序是 12、14、11、15、19，2022 到 2023 年下降，不是每年都上升。需要保存时使用 outputs/profit-trend.png。

In [ ]:
# 有需要再处理逐年趋势，不必为了跑完整份素材库而加做。


## 本次收尾：只记录今天真正用到的部分

请用自己的话填两三句，不需要回答整份素材库的所有题目。

- 我今天想解决的问题是：……
- 我做了什么，现在得到什么结果：……
- 我怎么核对这个结果：……
- 还需要帮忙的地方：……

保存后，从干净 kernel 重跑本次使用的模块及其前置步骤。完整成品可以另存精简 notebook，只保留这次需要的内容。

真实作业、私人数据与输出不要直接提交到公开 repo。合成数据只用于练习，不能拿来解释真实商店为什么赚钱。